# Data Access

# Data Reading

**Importing Libraries**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Reading CSV DATA**

**Trip Type Data**

In [0]:
storage_account = "project02adls"
storage_key = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
bronze = f"abfss://bronze@{storage_account}.dfs.core.windows.net"

In [0]:
silver = f"abfss://silver@{storage_account}.dfs.core.windows.net"

In [0]:
df_zone = spark.read.format("csv") \
                    .option("recursiveFileLookup", "true")\
                        .load(f"{bronze}/trip_type")


In [0]:
df_zone = spark.read.format("csv") \
                    .option("recursiveFileLookup", "true")\
                        .load(f"{bronze}/trip_zone")

In [0]:
display(df_zone)

In [0]:
df_trip_type=spark.read.csv(f"{bronze}/trip_type/trip_type.csv")
display(df_trip_type)

In [0]:
df_trip_type=spark.read.csv(f"{bronze}/trip_type/trip_type.csv")
display(df_trip_type)

In [0]:
df_trip_zone=spark.read.csv(f"{bronze}/trip_zone/taxi_zone_lookup.csv",header = True, inferSchema=True)
display(df_trip_zone)

In [0]:
df_trip_type = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{bronze}/trip_type/trip_type.csv")

In [0]:
df_trip_type=spark.read.csv(f"{bronze}/trip_type/trip_type.csv")
display(df_trip_type)

In [0]:
display(df_trip_type)

**Trip Zone**

In [0]:
# df_trip_zone = spark.read.format('csv')\
#                     .option('inferSchema',True)\
#                     .option('header',True)\
#                     .load('abfss://bronze@nyctaxistorageansh.dfs.core.windows.net/trip_zone')

In [0]:
df_trip_zone.display()

**Trip Data**

In [0]:
myschema = '''
                VendorID BIGINT,
                lpep_pickup_datetime TIMESTAMP,
                lpep_dropoff_datetime TIMESTAMP,
                store_and_fwd_flag STRING,
                RatecodeID BIGINT,
                PULocationID BIGINT,
                DOLocationID BIGINT,
                passenger_count BIGINT,
                trip_distance DOUBLE,
                fare_amount DOUBLE,
                extra DOUBLE,
                mta_tax DOUBLE,
                tip_amount DOUBLE,
                tolls_amount DOUBLE,
                ehail_fee DOUBLE,
                improvement_surcharge DOUBLE,
                total_amount DOUBLE,
                payment_type BIGINT,
                trip_type BIGINT,
                congestion_surcharge DOUBLE

      '''

In [0]:
df_trip=spark.read.parquet(f"{bronze}/trip-data/",schema=myschema)

In [0]:
df_trip = spark.read.format('parquet')\
              .schema(myschema)\
              .option('header',True)\
              .option('recursiveFileLookup',True)\
              .load(f"{bronze}/trip-data")

In [0]:
df_trip.display()

# Data Transformation

**Taxi Trip Type**

In [0]:
df_trip_type.display()

In [0]:
df_trip_type = df_trip_type.withColumnRenamed('description','trip_description')
df_trip_type.display()

In [0]:
df_trip_type.write.mode('append')\
               .parquet(f"{silver}/trip_type")

In [0]:
# df_trip_type.write.format('parquet')\
#             .mode('append')\
#             .option("path",f"{bronze}/trip_type")\
#             .save()

**Trip Zone**

In [0]:
df_trip_zone.display()

In [0]:
from pyspark.sql.functions import split, col

df_trip_zone = df_trip_zone.withColumn('zone1', split(col('Zone'), '/')[0]
                          ).withColumn('zone2', split(col('Zone'), '/')[1])

df_trip_zone.display()

In [0]:
df_trip_zone.write.mode('append').parquet(f"{silver}/trip_zone")

In [0]:
df_trip_zone.write.format('parquet')\
          .mode('append')\
          .option('path',f"{silver}/trip_zone")\
          .save()

In [0]:
df_check = spark.read.parquet(f"{silver}/trip_zone")
display(df_check)

In [0]:
print(df_trip)

**Trip Data**

In [0]:
df_trip.display()

In [0]:
df_trip = df_trip.withColumn('trip_date',to_date('lpep_pickup_datetime'))\
                  .withColumn('trip_year',year('lpep_pickup_datetime'))\
                  .withColumn('trip_month',month('lpep_pickup_datetime'))
                  

In [0]:
df_trip.display()

In [0]:
df_trip = df_trip.select('VendorID','PULocationID','DOLocationID','fare_amount','total_amount')
df_trip.display()

In [0]:
# df_trip.write.format('parquet')\
#             .mode('append')\
#             .option('path',f"{silver}/trip_data_2025")\
#             .save()

In [0]:
df_trip.write.mode('append').parquet(f"{silver}/trip_data_2025")

# Analysis

In [0]:
display(df_trip)

Databricks visualization. Run in Databricks to view.